# VisionSafe AI - Kaggle YOLO11 Full-Dataset Streaming Workflow
Bypasses Kaggle's 73GB disk limit and 12-hour limit by streaming the dataset in chunks.

In [ ]:
import time
SESSION_START_TIME = time.time()
TIME_LIMIT_SECONDS = 11.25 * 3600  # 11 hours and 15 minutes

!nvidia-smi
%pip install -q ultralytics==8.4.134 huggingface_hub pyyaml

In [ ]:
import os, json, yaml, shutil, random
from pathlib import Path
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

REPO_ID = 'iisc-aim/BMD-45'
TRAIN_ANN_FILE = 'BMD-45-Train/_annotations.coco.json'
VAL_ANN_FILE = 'BMD-45-Val/_annotations.coco.json'

print("Downloading COCO metadata...")
train_ann_path = hf_hub_download(repo_id=REPO_ID, filename=TRAIN_ANN_FILE, repo_type='dataset')
val_ann_path = hf_hub_download(repo_id=REPO_ID, filename=VAL_ANN_FILE, repo_type='dataset')

with open(train_ann_path, 'r') as f: train_coco = json.load(f)
with open(val_ann_path, 'r') as f: val_coco = json.load(f)

all_categories = sorted(train_coco['categories'], key=lambda x: x['id'])
category_map = {cat['id']: idx for idx, cat in enumerate(all_categories)}
category_names = [cat['name'] for cat in all_categories]
print(f"Target Classes: {category_names}")

In [ ]:
CHUNK_SIZE = 15000  # Images per chunk
EPOCHS_PER_CHUNK = 3

train_images_list = sorted(train_coco['images'], key=lambda x: x['id'])
random.seed(42)
random.shuffle(train_images_list)

dataset_dir = Path('/kaggle/working/dataset')
dataset_dir.mkdir(parents=True, exist_ok=True)  # FIX: CREATE THIS IMMEDIATELY

train_img_dir = dataset_dir / 'images' / 'train'
train_lbl_dir = dataset_dir / 'labels' / 'train'

def clean_directory(dir_path):
    if dir_path.exists():
        shutil.rmtree(dir_path)
    dir_path.mkdir(parents=True, exist_ok=True)

def convert_to_yolo(coco_data, image_chunk, target_img_dir, target_lbl_dir):
    img_id_to_file = {img['id']: img for img in image_chunk}
    downloaded = set([p.name for p in target_img_dir.glob('*.*')])
    
    for ann in coco_data['annotations']:
        if 'bbox' not in ann or len(ann['bbox']) != 4: continue
        img_info = img_id_to_file.get(ann['image_id'])
        if not img_info: continue
        
        filename = Path(img_info['file_name']).name
        if filename not in downloaded: continue
            
        cat_id = ann['category_id']
        if cat_id not in category_map: continue
            
        cls_id = category_map[cat_id]
        x, y, w, h = ann['bbox']
        dw, dh = 1.0 / img_info['width'], 1.0 / img_info['height']
        xc, yc, w, h = (x + w/2)*dw, (y + h/2)*dh, w*dw, h*dh
        
        if 0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1:
            label_path = target_lbl_dir / (Path(filename).stem + '.txt')
            with open(label_path, 'a') as f:
                f.write(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

In [ ]:
data_yaml_path = str(dataset_dir / 'data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump({'path': str(dataset_dir), 'train': 'images/train', 'val': 'images/train', 'nc': len(category_names), 'names': category_names}, f)

project_dir = '/kaggle/working/runs'
run_name = 'visionsafe_streaming'
checkpoint_dir = Path('/kaggle/working/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
state_file = checkpoint_dir / 'checkpoint_state.json'

total_chunks = (len(train_images_list) + CHUNK_SIZE - 1) // CHUNK_SIZE
start_chunk_idx = 0
current_weights = 'yolo11n.pt'

# Search for existing checkpoints to resume from (working dir or attached inputs)
candidate_state_files = [state_file] + list(Path('/kaggle/input').glob('**/checkpoint_state.json'))
for cand in candidate_state_files:
    if cand.exists():
        try:
            with open(cand, 'r') as f:
                saved_state = json.load(f)
            last_chunk = saved_state.get('completed_chunk_idx', -1)
            saved_weights = saved_state.get('weights_path', '')
            
            weights_to_use = None
            if saved_weights and Path(saved_weights).exists():
                weights_to_use = saved_weights
            elif (cand.parent / 'last.pt').exists():
                weights_to_use = str(cand.parent / 'last.pt')
            elif (cand.parent / 'best.pt').exists():
                weights_to_use = str(cand.parent / 'best.pt')
                
            if weights_to_use and last_chunk >= 0:
                start_chunk_idx = last_chunk + 1
                local_resume = checkpoint_dir / 'last.pt'
                if str(weights_to_use) != str(local_resume):
                    shutil.copy(weights_to_use, local_resume)
                current_weights = str(local_resume)
                print(f"\n>>> [CHECKPOINT RESUME] Resuming from Chunk {start_chunk_idx + 1}/{total_chunks}")
                print(f">>> Loaded previous checkpoint weights: {current_weights}")
                break
        except Exception as e:
            print(f"Notice reading checkpoint candidate {cand}: {e}")

if start_chunk_idx == 0:
    print(f"\n>>> [FRESH START] Starting training from Chunk 1/{total_chunks} with {current_weights}")

for chunk_idx in range(start_chunk_idx, total_chunks):
    if time.time() - SESSION_START_TIME > TIME_LIMIT_SECONDS:
        print(f"\nTime limit approaching! Gracefully pausing at chunk {chunk_idx + 1}. Checkpoint is preserved.")
        break
        
    start_img = chunk_idx * CHUNK_SIZE
    end_img = min((chunk_idx + 1) * CHUNK_SIZE, len(train_images_list))
    chunk = train_images_list[start_img:end_img]
    
    print(f"\n=== Processing Chunk {chunk_idx + 1}/{total_chunks} (Images {start_img} to {end_img}) ===")
    
    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            clean_directory(train_img_dir)
            clean_directory(train_lbl_dir)
            
            # Remove cache so YOLO rescans the new chunk
            if (dataset_dir / 'labels' / 'train.cache').exists():
                (dataset_dir / 'labels' / 'train.cache').unlink()
                
            print(f"Downloading {len(chunk)} images (Attempt {attempt}/{max_retries})...")
            for img_meta in chunk:
                try:
                    dest = train_img_dir / Path(img_meta['file_name']).name
                    dl_file = hf_hub_download(repo_id=REPO_ID, filename=f"BMD-45-Train/{img_meta['file_name']}", repo_type='dataset')
                    shutil.copy(dl_file, dest)
                except Exception:
                    pass
                    
            print("Converting labels...")
            convert_to_yolo(train_coco, chunk, train_img_dir, train_lbl_dir)
            
            print(f"Training on chunk {chunk_idx + 1} with weights {current_weights}...")
            model = YOLO(current_weights)
            model.train(
                data=data_yaml_path,
                epochs=EPOCHS_PER_CHUNK,
                imgsz=640,
                batch=-1,
                workers=4,
                project=project_dir,
                name=run_name,
                exist_ok=True,
                save=True
            )
            
            # Save chunk checkpoint
            latest_last = Path(project_dir) / run_name / 'weights' / 'last.pt'
            latest_best = Path(project_dir) / run_name / 'weights' / 'best.pt'
            
            if latest_last.exists():
                shutil.copy(latest_last, checkpoint_dir / 'last.pt')
                shutil.copy(latest_last, checkpoint_dir / f'checkpoint_chunk_{chunk_idx + 1}.pt')
                current_weights = str(checkpoint_dir / 'last.pt')
            if latest_best.exists():
                shutil.copy(latest_best, checkpoint_dir / 'best.pt')
                
            checkpoint_state = {
                'completed_chunk_idx': chunk_idx,
                'total_chunks': total_chunks,
                'images_processed': end_img,
                'total_images': len(train_images_list),
                'weights_path': str(checkpoint_dir / 'last.pt'),
                'best_weights_path': str(checkpoint_dir / 'best.pt') if latest_best.exists() else None,
                'timestamp': time.time(),
                'status': 'in_progress'
            }
            with open(state_file, 'w') as f:
                json.dump(checkpoint_state, f, indent=2)
                
            print(f">>> [CHECKPOINT SAVED] Chunk {chunk_idx + 1}/{total_chunks} progress saved to {state_file}")
            break
            
        except Exception as e:
            print(f"Warning: Chunk {chunk_idx + 1} attempt {attempt} failed: {e}")
            if attempt == max_retries:
                emergency_state = {
                    'last_successful_chunk': chunk_idx - 1,
                    'failed_chunk': chunk_idx,
                    'error': str(e),
                    'timestamp': time.time()
                }
                with open(checkpoint_dir / 'error_state.json', 'w') as f:
                    json.dump(emergency_state, f, indent=2)
                raise e
            time.sleep(10)

print("\nStreaming training complete! Packaging final artifacts...")
best_file = checkpoint_dir / 'best.pt'
if not best_file.exists():
    best_file = Path(project_dir) / run_name / 'weights' / 'best.pt'
if best_file.exists():
    shutil.make_archive('/kaggle/working/visionsafe_artifacts', 'zip', f"{project_dir}/{run_name}")
    print("Saved visionsafe_artifacts.zip with best checkpoint weights.")
